# Node2Vec (Weighted vs Unweighted)

- PageRank Weight
- 80/20 hold-out based on repos
- Hard Negative Sampling
- PecanPy SparseOTF Random Walk
- Gensim Word2Vec
- Precision@K / Recall@K

In [1]:
import gc
import json
import math
import os
import random
import tempfile
from collections import defaultdict
from itertools import combinations
from typing import Dict, Iterable, List, Sequence, Tuple

import numpy as np
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from pecanpy import pecanpy
from tqdm.auto import tqdm

In [39]:
RANDOM_SEED = 27
EMBEDDING_DIM = 128
NUM_WALKS = 10
WALK_LENGTH = 80
P = 1.0
Q = 1.0
WORKERS = max(1, (os.cpu_count() or 2) - 1)
RECOMMENDER = None

DEBUG_REPO_SAMPLE_RATIO = 0.05

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [54]:
def _iter_repo_records(path: str) -> Iterable[dict]:

    with open(path, "r", encoding="utf-8") as f:
        first_char = ""
        while True:
            ch = f.read(1)
            if not ch:
                break
            if not ch.isspace():
                first_char = ch
                break

        f.seek(0)
        if first_char == "[":
            data = json.load(f)
            for row in data:
                yield row
        else:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                yield json.loads(line)


def _get_repo_name(row: dict, fallback_idx: int) -> str:

    for key in ("repo_name", "full_name", "repo", "name", "repository"):
        value = row.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return f"__repo_{fallback_idx}"


def _parse_contributor_count(row: dict, contributors: Sequence[str]) -> int:

    raw_value = row.get("contributor_count")
    if raw_value is None:
        return len(contributors)
    try:
        return int(raw_value)
    except (TypeError, ValueError):
        return len(contributors)


def _extract_repo_contributors(repo_map_path: str) -> List[Tuple[str, List[str], int]]:

    repo_records: List[Tuple[str, List[str], int]] = []
    for idx, row in enumerate(tqdm(_iter_repo_records(repo_map_path), desc="Reading repos", unit="repo")):
        contributors = row.get("contributors", [])
        if not contributors:
            continue
        unique_contributors = list(dict.fromkeys(contributors))
        if len(unique_contributors) < 2:
            continue
        contributor_count = _parse_contributor_count(row, unique_contributors)
        if contributor_count < 2:
            continue
        repo_name = _get_repo_name(row, idx)
        repo_records.append((repo_name, unique_contributors, contributor_count))
    return repo_records


def _split_train_test_repos(
    repo_records: Sequence[Tuple[str, List[str], int]],
    test_size: float = 0.2,
    sample_ratio: float = 1.0,
    random_state: int = RANDOM_SEED,
) -> Tuple[List[Tuple[str, List[str], int]], List[Tuple[str, List[str], int]]]:

    if not repo_records:
        return [], []
    if not (0 < sample_ratio <= 1.0):
        raise ValueError("sample_ratio must be in (0, 1].")

    rng = random.Random(random_state)
    sampled = list(repo_records)
    if sample_ratio < 1.0:
        sampled = [r for r in sampled if rng.random() <= sample_ratio]
        if len(sampled) < 10:
            raise RuntimeError("Too few repos after sampling; increase sample_ratio.")

    rng.shuffle(sampled)
    split_idx = max(1, int(len(sampled) * (1.0 - test_size)))
    split_idx = min(split_idx, len(sampled) - 1)
    train_repos = sampled[:split_idx]
    test_repos = sampled[split_idx:]
    return train_repos, test_repos


def _build_weighted_edges_from_repos(
    repos: Sequence[Tuple[str, List[str], int]]
) -> Tuple[List[Tuple[str, str, float]], set, Dict[str, set], set]:

    edge_weights: Dict[Tuple[str, str], float] = defaultdict(float)
    train_adj: Dict[str, set] = defaultdict(set)
    all_users: set = set()

    for _, contributors, contributor_count in tqdm(repos, desc="Building train graph", unit="repo"):
        weight = 1.0 / math.log(contributor_count + 1)
        for u, v in combinations(contributors, 2):
            if u > v:
                u, v = v, u
            edge_weights[(u, v)] += weight
            train_adj[u].add(v)
            train_adj[v].add(u)
        all_users.update(contributors)

    edges = [(u, v, w) for (u, v), w in edge_weights.items()]
    train_pairs = set(edge_weights.keys())
    return edges, all_users, train_adj, train_pairs


def _extract_test_positive_pairs(
    test_repos: Sequence[Tuple[str, List[str], int]],
    train_users: set,
    train_pairs: set,
) -> Dict[str, set]:

    positives_by_user: Dict[str, set] = defaultdict(set)

    for _, contributors, _ in tqdm(test_repos, desc="Extracting test positives", unit="repo"):
        valid_users = [u for u in contributors if u in train_users]
        if len(valid_users) < 2:
            continue
        for u, v in combinations(valid_users, 2):
            if u > v:
                u, v = v, u
            if (u, v) in train_pairs:
                continue
            positives_by_user[u].add(v)
            positives_by_user[v].add(u)

    return positives_by_user


def prepare_repo_based_split(
    repo_map_path: str,
    sample_ratio: float = 1.0,
    test_size: float = 0.2,
    random_state: int = RANDOM_SEED,
):

    print("[1/7] Repo-based split and train graph construction...")
    if sample_ratio < 1.0:
        print(f"  sampling mode enabled: sample_ratio={sample_ratio:.2%}")

    repo_records = _extract_repo_contributors(repo_map_path)
    train_repos, test_repos = _split_train_test_repos(
        repo_records,
        test_size=test_size,
        sample_ratio=sample_ratio,
        random_state=random_state,
    )

    train_edges, all_users, train_adj, train_pairs = _build_weighted_edges_from_repos(train_repos)
    test_pos_by_user = _extract_test_positive_pairs(test_repos, all_users, train_pairs)

    n_pos_pairs = sum(len(vs) for vs in test_pos_by_user.values()) // 2
    print(
        f"  done: repos_total={len(repo_records)}, repos_train={len(train_repos)}, repos_test={len(test_repos)}, "
        f"users={len(all_users)}, train_edges={len(train_edges)}, test_positive_pairs={n_pos_pairs}"
    )

    return train_edges, all_users, train_adj, train_pairs, test_pos_by_user


def build_collab_pair_set(edges: Sequence[Tuple[str, str, float]]) -> set:
    pairs = set()
    for u, v, _ in edges:
        if u <= v:
            pairs.add((u, v))
        else:
            pairs.add((v, u))
    return pairs


def write_edgelist_file(edges: Sequence[Tuple[str, str, float]], force_unweighted: bool = False) -> str:
    tmp = tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", delete=False, suffix=".edg")
    with tmp as f:
        for u, v, w in tqdm(edges, total=len(edges), desc="Writing edgelist", unit="edge"):
            ww = 1.0 if force_unweighted else float(w)
            f.write(f"{u}\t{v}\t{ww}\n")
    return tmp.name


def run_node2vec_walks_with_sparseotf(edgelist_path: str) -> List[List[str]]:
    graph = pecanpy.SparseOTF(p=P, q=Q, workers=WORKERS, verbose=True)
    graph.read_edg(edgelist_path, weighted=True, directed=False)
    walks = graph.simulate_walks(num_walks=NUM_WALKS, walk_length=WALK_LENGTH)
    return walks


class TqdmEpochCallback(CallbackAny2Vec):

    def __init__(self, total_epochs: int, desc: str = "Training Word2Vec") -> None:
        self.total_epochs = total_epochs
        self.desc = desc
        self.pbar = None

    def on_train_begin(self, model) -> None:
        self.pbar = tqdm(total=self.total_epochs, desc=self.desc, unit="epoch")

    def on_epoch_end(self, model) -> None:
        self.pbar.update(1)

    def on_train_end(self, model) -> None:
        self.pbar.close()


def train_word2vec(
    walks: List[List[str]],
    desc: str = "Training Word2Vec",
    epochs: int = 5,
) -> Word2Vec:
    model = Word2Vec(
        sentences=walks,
        vector_size=EMBEDDING_DIM,
        window=10,
        min_count=0,
        sg=1,
        workers=WORKERS,
        epochs=epochs,
        seed=RANDOM_SEED,
        callbacks=[TqdmEpochCallback(total_epochs=epochs, desc=desc)],
    )
    return model


def cosine_score(model: Word2Vec, u: str, v: str) -> float:
    if u not in model.wv or v not in model.wv:
        return 0.0
    return float(model.wv.similarity(u, v))


def _sample_hard_negatives_for_user(
    user: str,
    pos_set: set,
    train_adj: Dict[str, set],
    all_users: Sequence[str],
    n_neg: int,
    rng: random.Random,
) -> List[str]:

    direct_neighbors = train_adj.get(user, set())

    second_hop = set()
    for nei in direct_neighbors:
        second_hop.update(train_adj.get(nei, set()))

    invalid = set(direct_neighbors)
    invalid.update(pos_set)
    invalid.add(user)
    second_hop = [x for x in second_hop if x not in invalid]
    rng.shuffle(second_hop)

    negatives: List[str] = second_hop[:n_neg]
    if len(negatives) >= n_neg:
        return negatives

    needed = n_neg - len(negatives)
    candidates = [u for u in all_users if u not in invalid and u not in set(negatives)]
    if not candidates:
        return negatives

    if len(candidates) <= needed:
        negatives.extend(candidates)
    else:
        negatives.extend(rng.sample(candidates, needed))

    return negatives


def evaluate_with_hard_negatives(
    model: Word2Vec,
    test_pos_by_user: Dict[str, set],
    train_adj: Dict[str, set],
    all_users: Sequence[str],
    n_eval_users: int = 1000,
    n_pos_per_user: int = 2,
    n_neg_per_pos: int = 50,
    random_state: int = RANDOM_SEED,
) -> Dict[str, float]:

    rng = random.Random(random_state)

    eval_users = [u for u, pos in test_pos_by_user.items() if pos and u in model.wv]
    if len(eval_users) > n_eval_users:
        eval_users = rng.sample(eval_users, n_eval_users)

    if not eval_users:
        return {"precision@1": 0.0, "precision@3": 0.0, "precision@5": 0.0, "recall@5": 0.0, "n_eval_users": 0, "n_eval_pairs": 0}

    p1_list: List[float] = []
    p3_list: List[float] = []
    p5_list: List[float] = []
    r5_list: List[float] = []
    n_eval_pairs = 0

    for user in tqdm(eval_users, desc="Hard-negative eval", unit="user"):
        user_vec = model.wv[user]
        pos_candidates = [p for p in test_pos_by_user[user] if p in model.wv]
        if not pos_candidates:
            continue

        if len(pos_candidates) > n_pos_per_user:
            pos_candidates = rng.sample(pos_candidates, n_pos_per_user)

        negatives = _sample_hard_negatives_for_user(
            user=user,
            pos_set=test_pos_by_user[user],
            train_adj=train_adj,
            all_users=all_users,
            n_neg=n_neg_per_pos,
            rng=rng,
        )
        negatives = [n for n in negatives if n in model.wv and n not in set(pos_candidates)]
        if not negatives:
            continue

        # 候选池固定为 n_pos_per_user 正样本 + n_neg_per_pos 负样本（不足时按可得数量继续评估）
        candidates = pos_candidates + negatives[:n_neg_per_pos]
        labels = np.zeros(len(candidates), dtype=np.int8)
        labels[: len(pos_candidates)] = 1

        mat = np.stack([model.wv[c] for c in candidates], axis=0)
        scores = np.dot(mat, user_vec)
        ranking = np.argsort(-scores)
        ranked_labels = labels[ranking]

        k1 = min(1, len(ranked_labels))
        k3 = min(3, len(ranked_labels))
        k5 = min(5, len(ranked_labels))

        p1_list.append(float(ranked_labels[:k1].sum() / k1))
        p3_list.append(float(ranked_labels[:k3].sum() / k3))
        p5_list.append(float(ranked_labels[:k5].sum() / k5))
        r5_list.append(float(ranked_labels[:k5].sum() / len(pos_candidates)))
        n_eval_pairs += 1

    def _safe_mean(xs: List[float]) -> float:
        return float(np.mean(xs)) if xs else 0.0

    return {
        "precision@1": _safe_mean(p1_list),
        "precision@3": _safe_mean(p3_list),
        "precision@5": _safe_mean(p5_list),
        "recall@5": _safe_mean(r5_list),
        "n_eval_users": len(eval_users),
        "n_eval_pairs": n_eval_pairs,
    }


class WeightedNode2VecRecommender:
    def __init__(
        self,
        model: Word2Vec,
        all_users: Sequence[str],
        collaborated_pairs: set,
    ) -> None:
        self.model = model
        self.all_users = list(all_users)
        self.collaborated_pairs = collaborated_pairs

    def recommend_developers(self, target_user: str, k: int = 3) -> List[Tuple[str, float]]:
        if target_user not in self.model.wv:
            return []

        recs = []
        for candidate in self.all_users:
            if candidate == target_user:
                continue
            pair = (target_user, candidate) if target_user <= candidate else (candidate, target_user)
            if pair in self.collaborated_pairs:
                continue
            if candidate not in self.model.wv:
                continue
            score = cosine_score(self.model, target_user, candidate)
            recs.append((candidate, score))

        recs.sort(key=lambda x: x[1], reverse=True)
        return recs[:k]


def recommend_developers(target_user: str, k: int = 3) -> List[Tuple[str, float]]:
    if RECOMMENDER is None:
        raise RuntimeError("Model is not initialized. Run the pipeline cell first.")
    return RECOMMENDER.recommend_developers(target_user=target_user, k=k)

In [41]:
repo_map_path = "repo_cleaned.json"
if not os.path.exists(repo_map_path):
    raise FileNotFoundError(f"{repo_map_path} not found in current working directory.")

(
    train_edges,
    all_users,
    train_adj,
    train_pairs,
    test_pos_by_user,
) = prepare_repo_based_split(
    repo_map_path,
    sample_ratio=DEBUG_REPO_SAMPLE_RATIO,
    test_size=0.2,
    random_state=RANDOM_SEED,
)

if len(train_edges) < 10:
    raise RuntimeError("Too few train edges after processing; cannot run reliable evaluation.")

n_test_pos_pairs = sum(len(vs) for vs in test_pos_by_user.values()) // 2
print(
    f"sample_ratio={DEBUG_REPO_SAMPLE_RATIO:.2%}, "
    f"n_users={len(all_users)}, n_train_edges={len(train_edges)}, "
    f"n_test_positive_pairs={n_test_pos_pairs}"
)

[1/7] Repo-based split and train graph construction...
  sampling mode enabled: sample_ratio=5.00%


Reading repos: 0repo [00:00, ?repo/s]

Building train graph:   0%|          | 0/123656 [00:00<?, ?repo/s]

Extracting test positives:   0%|          | 0/30915 [00:00<?, ?repo/s]

  done: repos_total=3083150, repos_train=123656, repos_test=30915, users=329780, train_edges=2180810, test_positive_pairs=21117
sample_ratio=5.00%, n_users=329780, n_train_edges=2180810, n_test_positive_pairs=21117


In [55]:
all_pairs = train_pairs

n_eval_users_total = len([u for u, vs in test_pos_by_user.items() if len(vs) > 0])
print(
    f"train_pairs={len(all_pairs)}, users_with_test_positive={n_eval_users_total}, "
    f"hard_neg_per_positive=50"
)

train_pairs=2180810, users_with_test_positive=5125, hard_neg_per_positive=50


In [44]:
weighted_edg_file = write_edgelist_file(train_edges, force_unweighted=False)
weighted_walks = run_node2vec_walks_with_sparseotf(weighted_edg_file)
os.remove(weighted_edg_file)

print(f"weighted_walks={len(weighted_walks)}")

Writing edgelist:   0%|          | 0/2180810 [00:00<?, ?edge/s]

  0%|          | 0/3297800 [00:00<?, ?it/s]

weighted_walks=3297800


In [45]:
unweighted_edg_file = write_edgelist_file(train_edges, force_unweighted=True)
unweighted_walks = run_node2vec_walks_with_sparseotf(unweighted_edg_file)
os.remove(unweighted_edg_file)

print(f"unweighted_walks={len(unweighted_walks)}")

Writing edgelist:   0%|          | 0/2180810 [00:00<?, ?edge/s]

  0%|          | 0/3297800 [00:00<?, ?it/s]

unweighted_walks=3297800


In [46]:
weighted_model = train_word2vec(weighted_walks, desc="Weighted Word2Vec")
del weighted_walks
gc.collect()

print("weighted_model trained")

Weighted Word2Vec:   0%|          | 0/5 [00:00<?, ?epoch/s]

weighted_model trained


In [47]:
unweighted_model = train_word2vec(unweighted_walks, desc="Unweighted Word2Vec")
del unweighted_walks
gc.collect()

print("unweighted_model trained")

Unweighted Word2Vec:   0%|          | 0/5 [00:00<?, ?epoch/s]

unweighted_model trained


In [56]:
N_EVAL_USERS = globals().get("EVAL_N_USERS_OVERRIDE", 1000)
N_NEG_PER_POS = globals().get("EVAL_N_NEG_PER_POS_OVERRIDE", 50)

weighted_metrics = evaluate_with_hard_negatives(
    model=weighted_model,
    test_pos_by_user=test_pos_by_user,
    train_adj=train_adj,
    all_users=list(all_users),
    n_eval_users=N_EVAL_USERS,
    n_pos_per_user=2,
    n_neg_per_pos=N_NEG_PER_POS,
    random_state=RANDOM_SEED,
)

unweighted_metrics = evaluate_with_hard_negatives(
    model=unweighted_model,
    test_pos_by_user=test_pos_by_user,
    train_adj=train_adj,
    all_users=list(all_users),
    n_eval_users=N_EVAL_USERS,
    n_pos_per_user=2,
    n_neg_per_pos=N_NEG_PER_POS,
    random_state=RANDOM_SEED,
)

def _improvement(a, b):
    return ((a - b) / b * 100.0) if b != 0 else float("inf")

p1_imp = _improvement(weighted_metrics["precision@1"], unweighted_metrics["precision@1"])
p3_imp = _improvement(weighted_metrics["precision@3"], unweighted_metrics["precision@3"])
p5_imp = _improvement(weighted_metrics["precision@5"], unweighted_metrics["precision@5"])
r5_imp = _improvement(weighted_metrics["recall@5"], unweighted_metrics["recall@5"])

print("\n===== Node2Vec Link Prediction Comparison (Hard Negatives) =====")
print(f"{'Metric':<14} {'Weighted':>12} {'Unweighted':>12} {'Improvement':>12}")
print("-" * 54)
print(
    f"{'Precision@1':<14} {weighted_metrics['precision@1']:>12.6f} "
    f"{unweighted_metrics['precision@1']:>12.6f} {p1_imp:>11.2f}%"
)
print(
    f"{'Precision@3':<14} {weighted_metrics['precision@3']:>12.6f} "
    f"{unweighted_metrics['precision@3']:>12.6f} {p3_imp:>11.2f}%"
)
print(
    f"{'Precision@5':<14} {weighted_metrics['precision@5']:>12.6f} "
    f"{unweighted_metrics['precision@5']:>12.6f} {p5_imp:>11.2f}%"
)
print(
    f"{'Recall@5':<14} {weighted_metrics['recall@5']:>12.6f} "
    f"{unweighted_metrics['recall@5']:>12.6f} {r5_imp:>11.2f}%"
)

print(
    f"\nn_eval_users={N_EVAL_USERS}, hard_neg_per_positive={N_NEG_PER_POS}, "
    f"weighted_eval_pairs={weighted_metrics['n_eval_pairs']}, "
    f"unweighted_eval_pairs={unweighted_metrics['n_eval_pairs']}"
)

results = {
    "weighted_precision_at_1": weighted_metrics["precision@1"],
    "unweighted_precision_at_1": unweighted_metrics["precision@1"],
    "weighted_precision_at_3": weighted_metrics["precision@3"],
    "unweighted_precision_at_3": unweighted_metrics["precision@3"],
    "weighted_precision_at_5": weighted_metrics["precision@5"],
    "unweighted_precision_at_5": unweighted_metrics["precision@5"],
    "weighted_recall_at_5": weighted_metrics["recall@5"],
    "unweighted_recall_at_5": unweighted_metrics["recall@5"],
    "precision_at_1_improvement_pct": p1_imp,
    "precision_at_3_improvement_pct": p3_imp,
    "precision_at_5_improvement_pct": p5_imp,
    "recall_at_5_improvement_pct": r5_imp,
    "n_eval_users": N_EVAL_USERS,
    "n_neg_per_positive": N_NEG_PER_POS,
    "n_users": len(all_users),
    "n_train_edges": len(train_edges),
}
results

Hard-negative eval:   0%|          | 0/1000 [00:00<?, ?user/s]

Hard-negative eval:   0%|          | 0/1000 [00:00<?, ?user/s]


===== Node2Vec Link Prediction Comparison (Hard Negatives) =====
Metric             Weighted   Unweighted  Improvement
------------------------------------------------------
Precision@1        0.112000     0.099000       13.13%
Precision@3        0.074000     0.065000       13.85%
Precision@5        0.059200     0.053800       10.04%
Recall@5           0.193500     0.172000       12.50%

n_eval_users=1000, hard_neg_per_positive=50, weighted_eval_pairs=1000, unweighted_eval_pairs=1000


{'weighted_precision_at_1': 0.112,
 'unweighted_precision_at_1': 0.099,
 'weighted_precision_at_3': 0.074,
 'unweighted_precision_at_3': 0.065,
 'weighted_precision_at_5': 0.0592,
 'unweighted_precision_at_5': 0.0538,
 'weighted_recall_at_5': 0.1935,
 'unweighted_recall_at_5': 0.172,
 'precision_at_1_improvement_pct': 13.131313131313128,
 'precision_at_3_improvement_pct': 13.846153846153836,
 'precision_at_5_improvement_pct': 10.037174721189594,
 'recall_at_5_improvement_pct': 12.50000000000001,
 'n_eval_users': 1000,
 'n_neg_per_positive': 50,
 'n_users': 329780,
 'n_train_edges': 2180810}